In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, udf, when, size, split
from pyspark.sql.types import IntegerType, FloatType, StringType, StructType, StructField
from pyspark.ml.feature import VectorAssembler, StringIndexer, IndexToString
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml import Pipeline
import re
import requests
import numpy as np
import pandas as pd

In [ ]:
spark = SparkSession.builder \
    .appName("SmartCVScreener") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("✅ Spark running —", spark.version)

In [ ]:
np.random.seed(42)
n = 600000

df_big = pd.DataFrame({
    'skills_count': np.random.randint(1, 15, n),
    'years_clean':  np.random.randint(0, 20, n),
    'has_degree':   np.random.randint(0, 2, n),
})

def label(row):
    if row.skills_count >= 6 and row.years_clean >= 4 and row.has_degree == 1:
        return 'shortlist'
    elif row.skills_count <= 2 and row.years_clean <= 1:
        return 'reject'
    else:
        return 'review'

df_big['label'] = df_big.apply(label, axis=1)
df_big.to_csv('cv_big_dataset.csv', index=False)

print(f"✅ Dataset generated: {df_big.shape[0]:,} rows")
print(df_big['label'].value_counts())

In [ ]:
df = spark.read.csv(
    'cv_big_dataset.csv',
    header=True,
    inferSchema=True
)

df.printSchema()
print(f"Total rows: {df.count():,}")
df.show(5)

In [ ]:
# Convert label strings → numeric index
label_indexer = StringIndexer(
    inputCol="label",
    outputCol="label_idx",
    handleInvalid="skip"
)

# Assemble the 3 features into one vector
assembler = VectorAssembler(
    inputCols=["skills_count", "years_clean", "has_degree"],
    outputCol="features"
)

# Apply both
df_indexed  = label_indexer.fit(df).transform(df)
df_assembled = assembler.transform(df_indexed)

df_assembled.select("features", "label_idx", "label").show(5)

In [ ]:
from pyspark.ml.classification import RandomForestClassifier

train_df, test_df = df_assembled.randomSplit([0.8, 0.2], seed=42)

print(f"Training rows: {train_df.count():,}")
print(f"Test rows:     {test_df.count():,}")

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label_idx",
    numTrees=100,
    seed=42
)

model = rf.fit(train_df)
print("✅ Random Forest model trained")

In [ ]:
predictions = model.transform(test_df)

evaluator = MulticlassClassificationEvaluator(
    labelCol="label_idx",
    predictionCol="prediction",
    metricName="accuracy"
)

accuracy = evaluator.evaluate(predictions)
print(f"✅ Test Accuracy: {round(accuracy * 100, 2)}%")

predictions.select("label", "prediction").show(10)

In [ ]:
model.save("cv_spark_model")
print("✅ Random Forest model saved to ./cv_spark_model/")